# EDA · Kenya · perinatal (headline cohort)

Dataset: `data/processed/ke_perinatal.csv`.
Outcome: `perinatal_death` (stillbirth or death in the first 7 days).

Base predictors only. This is the main cohort for the headline result.

What this notebook should cover:
- shape and dtypes; confirm the row count matches the proposal
- missingness (expect blanks in told_high_bp, preceding_interval_m)
- the outcome rate
- distributions of the predictors (maternal_age, prior_pregnancies, prior_live_births, prior_stillbirth, prior_loss, preceding_interval_m, multiple, education, wealth_quintile, residence, region, told_high_bp) and the outcome rate within each category

Clear outputs before committing.


In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path

# Load the CSV file into a DataFrame
repo_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "data" / "processed" / "ke_perinatal.csv").exists()
)
df = pd.read_csv(repo_root / "data" / "processed" / "ke_perinatal.csv")

# Display the first five rows
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])

In [ ]:
# Data type
df.info()

In [ ]:
# Target
target = "perinatal_death"

# Base predictors for the headline perinatal model
base_predictors = [
    "maternal_age",
    "education",
    "wealth_quintile",
    "residence",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m",
    "multiple"
]

# Variables to explore in EDA only
eda_only = ["region","told_high_bp"]

# Helper columns (not model predictors)
helper_columns = ["weight","v001"]

# All columns required for this EDA
eda_columns = (
    [target]
    + base_predictors
    + eda_only
    + helper_columns
)

# Check that all required columns exist
missing_columns = [
    col for col in eda_columns
    if col not in df.columns
]

if not missing_columns:
    print("PASS: All required EDA columns are available.")
else:
    print("Missing columns:", missing_columns)


All required columns are available

The weighted average suppose we want to calculate the perinatal-death rate among mothers with previous stillbirths.

In [ ]:
weighted_rate = np.average(
    df["perinatal_death"],
    weights=df["weight"]
) * 100

Checking for any missing values

In [ ]:
missing_count = df[eda_columns].isnull().sum()
print(missing_count)

In [ ]:
missing_percentage = (df[eda_columns].isnull().mean() * 100)
print(missing_percentage.round(2))

preceding_interval_m has 4966 missing values, which is around 25.05% of the total observations & told_high_bp - 9383 (47.34%)

In [ ]:
target_counts = df[target].value_counts().sort_index()
print(target_counts)

0: 19,145 observations without perinatal death.

1: 677 observations with perinatal death.

Weighted and unweighted perinetal deaths

In [ ]:
# Calculate unweighted target rate - without applying survey weights

unweighted_rate = df[target].mean() * 100
print(f"Unweighted target rate: {unweighted_rate:.2f}%")

In [ ]:
# Calculate weighted target rate
weighted_rate = np.average(
    df[target],
    weights=df["weight"]
) * 100

print(f"Weighted target rate: {weighted_rate:.2f}%")

Unweighted target rate: 3.42% while Weighted target rate: 3.15%

Summary table

In [ ]:
total = len(df)
deaths = (df[target] == 1).sum()
survivals = (df[target] == 0).sum()

target_summary = pd.DataFrame({
    "Target Category": [
        "No perinatal death",
        "Perinatal death"
    ],
    "Count": [
        survivals,
        deaths
    ],
    "Unweighted Percentage": [
        survivals / total * 100,
        deaths / total * 100
    ]
})

display(target_summary.round(2))

print(f"Weighted target rate: {weighted_rate:.2f}%")

### Predictor Distribution

summary of the numeric predictors

In [ ]:
numeric_predictors = [
    "maternal_age",
    "prior_live_births",
    "prior_stillbirth",
    "prior_loss",
    "preceding_interval_m"
]

# Generate descriptive statistics
numeric_summary = df[numeric_predictors].describe().T

display(numeric_summary.round(2))

The descriptive analysis reveals that the average maternal age was 26.73 years (SD = 6.47), with a median of 26 years and an observed range of 11–47 years. 

Mothers had an average of 2.22 previous live births, while approximately 3% and 12% of observations had a recorded previous stillbirth and pregnancy loss, respectively.

The preceding birth interval had a mean of 43.88 months and a median of 35 months among the 14,856 observations with available data. The variable also contained an extreme maximum value of 309 months.

In [ ]:
# histograms for numerical predictors
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for column, ax in zip(numeric_predictors, axes.flat):
    sns.histplot(
        data=df,
        x=column,
        bins=30,
        kde=True,
        ax=ax
    )

    ax.set_title(column.replace("_", " ").title())
    ax.set_xlabel(column.replace("_", " ").title())
    ax.set_ylabel("Number of observations")

plt.tight_layout()
plt.show()

The categorical variables frequency deistribution

In [ ]:
categorical_predictors = [
    "education",
    "wealth_quintile",
    "residence",
    "multiple",
    "region",
    "told_high_bp"
]

In [ ]:
education_counts = df["education"].value_counts(dropna=False)
print(education_counts)


In [ ]:
# Function to calculate category counts and weighted percentages
def weighted_distribution(data, column):
    summary = data.groupby(
        column,
        dropna=False
    ).agg(
        Count=(column, "size"),
        Weighted_Total=("weight", "sum")
    )
    summary["Weighted_Percentage"] = (
        summary["Weighted_Total"]
        / summary["Weighted_Total"].sum()
        * 100
    )

    return summary.round(2)

# Test the function using education
education_summary = weighted_distribution(df, "education")
display(education_summary)


In [ ]:
# Categorical predictors & their summaries
categorical_predictors = [
    "education",
    "wealth_quintile",
    "residence",
    "multiple",
    "region",
    "told_high_bp"
]

# Generate frequency tables for all categorical variables
for column in categorical_predictors:
    print(f"\nDistribution of {column}")
    summary = weighted_distribution(df, column)
    display(summary)

In [ ]:
# Plot weighted distributions of categorical variables
for column in categorical_predictors:
    summary = weighted_distribution(df, column)
    summary = summary.sort_values("Weighted_Percentage",ascending=True)

    plt.figure(figsize=(8, 6))
    sns.barplot(x=summary["Weighted_Percentage"],y=summary.index.astype(str),color="steelblue")

    plt.title(f"Weighted Distribution of {column.replace('_', ' ').title()}")
    plt.xlabel("Weighted Percentage (%)")
    plt.ylabel(column.replace("_", " ").title())
    plt.tight_layout()
    plt.show()


primary education represented the largest weighted share of the cohort (38.01%), followed by secondary education (33.77%). The wealth distribution was relatively balanced across the five quintiles, with the poorest category accounting for the largest weighted share (22.45%). Rural residents represented 62.59% of the weighted population, compared with 37.41% from urban areas.

Most pregnancies were singleton pregnancies (96.71%), while multiple pregnancies accounted for 3.29%. The dataset included observations from all 47 Kenyan counties, although weighted representation varied geographically, with Nairobi accounting for the largest share (11.48%).

### Weighted target rates by predictor category

In [ ]:
def weighted_target_rate(data, column):

    # Create a temporary copy
    temp = data[[column, target, "weight"]].copy()

    # Retain missing predictor values as a separate category
    temp[column] = temp[column].astype("object").where(
        temp[column].notna(), "Missing"
    )

    # Group observations by predictor category
    summary = temp.groupby(column, dropna=False).apply(
        lambda group: pd.Series({
            "Count": len(group),
            "Deaths": (group[target] == 1).sum(),
            "Weighted_Total": group["weight"].sum(),
            "Weighted_Deaths": (
                group[target] * group["weight"]
            ).sum(),
            "Weighted_Target_Rate": (
                group[target] * group["weight"]
            ).sum() / group["weight"].sum() * 100
        }),
        include_groups=False
    )

    return summary.round(2)


In [ ]:
# Categorise maternal age
df["maternal_age_group"] = pd.cut(
    df["maternal_age"],
    bins=[0, 17, 24, 34, 100],
    labels=[
        "Under 18",
        "18-24",
        "25-34",
        "35+"
    ]
)

# Examine the age categories
print(df["maternal_age_group"].value_counts().sort_index())

In [ ]:
age_results = weighted_target_rate(
    df,
    "maternal_age_group"
)

display(age_results)

In [ ]:
# Target rates by number of previous live births

live_birth_results = weighted_target_rate(
    df,
    "prior_live_births"
)

display(live_birth_results)

In [ ]:
# Target rates by previous stillbirth history

stillbirth_results = weighted_target_rate(
    df,
    "prior_stillbirth"
)

display(stillbirth_results)

In [ ]:
# Target rates by previous pregnancy loss

loss_results = weighted_target_rate(
    df,
    "prior_loss"
)

display(loss_results)

In [ ]:
# Categorise preceding birth intervals

df["interval_group"] = pd.cut(
    df["preceding_interval_m"],
    bins=[0, 23, 35, 59, float("inf")],
    labels=[
        "Under 24 months",
        "24-35 months",
        "36-59 months",
        "60+ months"
    ]
)

# Check the distribution
print(df["interval_group"].value_counts(dropna=False))

In [ ]:
# Weighted target rates by preceding birth interval

interval_results = weighted_target_rate(
    df,
    "interval_group"
)

display(interval_results)

In [ ]:
# List of categorical predictors

categorical_predictors = [
    "education",
    "wealth_quintile",
    "residence",
    "multiple"
]

# Calculate weighted target rates

categorical_results = {}

for column in categorical_predictors:

    print(f"\nWeighted target rate by {column}")

    results = weighted_target_rate(df, column)

    categorical_results[column] = results

    display(results)


In [ ]:
# Weighted target rates by region
region_results = weighted_target_rate(
    df,
    "region"
)

# Display highest rates first
region_results = region_results.sort_values(
    by="Weighted_Target_Rate",
    ascending=False
)

display(region_results)

In [ ]:
# Weighted target rates by self-reported high blood pressure
bp_results = weighted_target_rate(
    df,
    "told_high_bp"
)

display(bp_results)

In [ ]:
# Plot weighted target rates by maternal age

plt.figure(figsize=(9, 5))

sns.barplot(
    data=age_results.reset_index(),
    x="maternal_age_group",
    y="Weighted_Target_Rate",
    color="steelblue"
)

plt.title("Weighted Perinatal Death Rate by Maternal Age")
plt.xlabel("Maternal Age Group")
plt.ylabel("Weighted Target Rate (%)")

plt.tight_layout()
plt.show()
